In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name
import json

source_path = "abfss://landing@stroutemindeuskadidev.dfs.core.windows.net/restaurants/restaurants.json"
delta_path = "abfss://bronze@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/gastronomy/data"
deltaTable = "dbw_routemind_euskadi_dev.bronze.gastronomy"



df_restaurants_raw = spark.read \
    .option("multiline", "true") \
    .json(source_path)


In [0]:
df_restaurants_bronze = df_restaurants_raw.withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name())

In [0]:
spark.sql(f"CREATE TABLE IF NOT EXISTS dbw_routemind_euskadi_dev.bronze.gastronomy USING DELTA LOCATION '{delta_path}'")

df_restaurants_bronze.write.option("path", delta_path).option("mergeSchema", True).saveAsTable(name=deltaTable, format="delta", mode="overwrite")


In [0]:
%sql
select * from dbw_routemind_euskadi_dev.bronze.cultural_places limit 2